In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# ── 1. 读取数据 ──────────────────────────────────────────────
df = pd.read_csv("marvel-unimodal-nodes.csv")

In [ ]:
# ── 2. 列名重命名，统一为简短变量名 ─────────────────────────
df = df.rename(columns={
    'team(正派 (Hero)、反派 (Villain)、中立 (Neutral) -主属性-只写英文）': 'alignment',
    'team（复仇者联盟、银河护卫队、神盾局）': 'team',
    'first_appearance（剧中）': 'first_appearance'
})

In [ ]:
# ── 3. 清洗 gender ───────────────────────────────────────────
df['gender'] = df['gender'].str.strip().str.title()
df['gender'] = df['gender'].replace({'Xiongxing': 'Unknown'})
df['gender'] = df['gender'].where(df['gender'].isin(['Male', 'Female']), other='Unknown')

In [ ]:
# ── 4. 清洗 alignment ────────────────────────────────────────
df['alignment'] = df['alignment'].str.strip().str.title()
# Anti-Hero 归入 Neutral（可按需修改）
df['alignment'] = df['alignment'].replace({'Anti-Hero': 'Neutral'})
df['alignment'] = df['alignment'].where(
    df['alignment'].isin(['Hero', 'Villain', 'Neutral']), other='Unknown'
)

In [ ]:
# ── 5. 清洗 race（合并细分值到 Other）───────────────────────
KNOWN_RACES = ['White', 'Black', 'Asian']
df['race'] = df['race'].str.strip().str.title()
df['race'] = df['race'].apply(lambda r: r if r in KNOWN_RACES else 'Other')

In [ ]:
# ── 6. 时代映射 ──────────────────────────────────────────────
DECADE_MAP = {
    '1930s': 1935, '1940s': 1945, '1950s': 1955,
    '1960s': 1965, '1970s': 1975, '1980s': 1985, '1990s': 1995
}

def map_era(val):
    val = str(val).strip()
    # 处理 "1960s" 等模糊年代
    if val in DECADE_MAP:
        year = DECADE_MAP[val]
    else:
        try:
            year = int(float(val))
        except (ValueError, TypeError):
            return 'Unknown Age'   # Unknown / Ancient times / 空值
    # 按标准划分
    if year <= 1961:
        return 'before 1961'
    elif 1961 < year <= 1970:
        return 'Silver Age'
    elif 1970 < year <= 1985:
        return 'Bronze Age'
    elif 1985 < year:
        return 'Modern Age'
    else:
        return 'Unknown Age'

df['era'] = df['first_appearance'].apply(map_era)

In [ ]:
# ── 7. team 归属数量 ─────────────────────────────────────────
def count_teams(val):
    val = str(val).strip()
    if val.lower() in ('', 'none', 'nan'):
        return 0
    return len([t for t in val.split(',') if t.strip()])

df['team_count'] = df['team'].apply(count_teams)

In [ ]:
# ── 8. 输出 Gephi 节点表 ─────────────────────────────────────
node_gephi = df[['Id', 'Label', 'gender', 'alignment', 'race', 'era', 'team_count']].copy()
node_gephi.to_csv("nodes_gephi.csv", index=False)
print("✅ nodes_gephi.csv 已保存")

In [ ]:
# ── 9. 4 时代汇总对比表 ──────────────────────────────────────
ERA_ORDER = ['before 1961', 'Silver Age', 'Bronze Age', 'Modern Age', 'Unknown Age']

rows = []
for era in ERA_ORDER:
    sub = df[df['era'] == era]
    row = {'Era': era, 'Total': len(sub)}
    for v in ['Male', 'Female', 'Unknown']:
        row[f'Gender_{v}'] = (sub['gender'] == v).sum()
    for v in ['Hero', 'Villain', 'Neutral']:
        row[f'Align_{v}'] = (sub['alignment'] == v).sum()
    for v in ['White', 'Black', 'Asian', 'Other']:
        row[f'Race_{v}'] = (sub['race'] == v).sum()
    row['Avg_Team_Count'] = round(sub['team_count'].mean(), 2) if len(sub) > 0 else 0
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Era')
summary_df.to_csv("era_summary_table.csv")
print("✅ era_summary_table.csv 已保存")
print(summary_df)

In [ ]:
# ── 10. 4 时代 node feature 对比图 ─────────────────
COMPARE_ERAS = ['before 1961', 'Silver Age', 'Bronze Age', 'Modern Age']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Node Feature Comparison Across 4 Eras', fontsize=15, fontweight='bold')

# --- 子图1：Gender 分组柱状图 ---
gender_cats = ['Male', 'Female', 'Unknown']
x = np.arange(len(COMPARE_ERAS))
width = 0.25
colors_g = ['steelblue', 'lightcoral', 'gray']

ax = axes[0]
for i, cat in enumerate(gender_cats):
    vals = [summary_df.loc[era, f'Gender_{cat}'] for era in COMPARE_ERAS]
    bars = ax.bar(x + i * width, vals, width, label=cat, color=colors_g[i])
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.3, str(v),
                    ha='center', va='bottom', fontsize=8)

ax.set_title('Gender', fontsize=12)
ax.set_xticks(x + width)
ax.set_xticklabels([e.replace(' Age','') for e in COMPARE_ERAS], fontsize=9)
ax.set_ylabel('Count')
ax.legend(fontsize=9)

# --- 子图2：Alignment 分组柱状图 ---
align_cats = ['Hero', 'Villain', 'Neutral']
colors_a = ['mediumseagreen', 'tomato', 'goldenrod']

ax = axes[1]
for i, cat in enumerate(align_cats):
    vals = [summary_df.loc[era, f'Align_{cat}'] for era in COMPARE_ERAS]
    bars = ax.bar(x + i * width, vals, width, label=cat, color=colors_a[i])
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.3, str(v),
                    ha='center', va='bottom', fontsize=8)

ax.set_title('Alignment', fontsize=12)
ax.set_xticks(x + width)
ax.set_xticklabels([e.replace(' Age','') for e in COMPARE_ERAS], fontsize=9)
ax.set_ylabel('Count')
ax.legend(fontsize=9)

# --- 子图3：Race 分组柱状图 ---
race_cats = ['White', 'Black', 'Asian', 'Other']
colors_r = ['#d4a574', '#4a4a4a', '#f4c542', '#88c57f']

ax = axes[2]
width2 = 0.2
for i, cat in enumerate(race_cats):
    vals = [summary_df.loc[era, f'Race_{cat}'] for era in COMPARE_ERAS]
    bars = ax.bar(x + i * width2, vals, width2, label=cat, color=colors_r[i])
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.3, str(v),
                    ha='center', va='bottom', fontsize=8)

ax.set_title('Race', fontsize=12)
ax.set_xticks(x + width2 * 1.5)
ax.set_xticklabels([e.replace(' Age','') for e in COMPARE_ERAS], fontsize=9)
ax.set_ylabel('Count')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("node_features_era_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ node_features_era_comparison.png 已保存")

In [ ]:
# ── 11. 4 张分时代统计图 ─────────────────────────────────────
PLOT_ERAS = ['before 1961', 'Silver Age', 'Bronze Age', 'Modern Age']

FEATURE_CFG = [
    ('gender',    ['Male', 'Female', 'Unknown'],         'steelblue',      'Gender'),
    ('alignment', ['Hero', 'Villain', 'Neutral', 'Unknown'], 'salmon',     'Alignment'),
    ('race',      ['White', 'Black', 'Asian', 'Other'],  'mediumseagreen', 'Race'),
]

for era in PLOT_ERAS:
    sub = df[df['era'] == era]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'{era}  (n={len(sub)})  — Node Feature Distribution',
                 fontsize=14, fontweight='bold')

    for ax, (feat, categories, color, title) in zip(axes, FEATURE_CFG):
        counts = sub[feat].value_counts().reindex(categories, fill_value=0)
        bars = ax.bar(counts.index, counts.values, color=color, edgecolor='white')
        ax.set_title(title, fontsize=12)
        ax.set_ylabel('Count')
        ax.set_ylim(0, max(counts.values.max() * 1.2, 1))
        for bar, v in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.2,
                    str(v), ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    fname = f"node_features_{era.replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ {fname} 已保存")